# Module 5.1: MIDAS Regression

## Learning Objectives

By completing this notebook, you will:

1. **Understand** the MIDAS (Mixed Data Sampling) regression framework
2. **Implement** Beta polynomial weighting schemes for high-frequency data
3. **Estimate** MIDAS models via nonlinear least squares
4. **Forecast** quarterly GDP using monthly indicators
5. **Compare** MIDAS to traditional frequency-matched approaches

## Prerequisites

- Time series regression (OLS, AR models)
- Nonlinear optimization basics
- Understanding of temporal aggregation (Module 5 Guide 1)

## Estimated Time: 90-120 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "the MIDAS (Mixed Data Sampling) regression framework",
    "Beta polynomial weighting schemes for high-frequency data",
    "MIDAS models via nonlinear least squares",
    "quarterly GDP using monthly indicators",
    "MIDAS to traditional frequency-matched approaches"
])

## Setup and Imports

We'll use standard scientific computing libraries plus optimization tools for MIDAS estimation.

In [ ]:
# Purpose: Import required libraries for MIDAS regression
# Key Concept: MIDAS requires optimization for Beta polynomial parameters

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---

# Part 1: MIDAS Weighting Schemes

## The MIDAS Approach

MIDAS regression forecasts a low-frequency variable (e.g., quarterly GDP) using high-frequency predictors (e.g., monthly indicators) **without aggregating** the high-frequency data.

**Basic MIDAS specification:**
$$y_t^{(q)} = \beta_0 + \beta_1 \sum_{j=0}^{J-1} w(j; \theta) x_{t-j}^{(m)} + \varepsilon_t$$

where:
- $y_t^{(q)}$: low-frequency target (e.g., quarterly GDP growth)
- $x_{t-j}^{(m)}$: high-frequency predictor at lag $j$ (e.g., monthly industrial production)
- $w(j; \theta)$: data-driven weighting function
- $J$: number of high-frequency lags to include

**Key insight:** The weighting function $w(j; \theta)$ is parsimoniously parameterized (few parameters control all weights), avoiding over-parameterization.

## 1.1 Beta Polynomial Weights

The most popular weighting scheme uses the **Beta probability density function**:

$$w(j; \theta_1, \theta_2) = \frac{(1-j/J)^{\theta_1-1}(j/J)^{\theta_2-1}}{\sum_{k=0}^{J-1}(1-k/J)^{\theta_1-1}(k/J)^{\theta_2-1}}$$

where $\theta_1, \theta_2 > 0$ control the shape:
- $\theta_1 > 1, \theta_2 < 1$: Recent lags weighted more heavily
- $\theta_1 < 1, \theta_2 > 1$: Distant lags weighted more heavily  
- $\theta_1 = \theta_2$: Symmetric, hump-shaped weights

In [ ]:
# Purpose: Implement Beta polynomial weighting function
# Key Concept: Two parameters control entire lag distribution

def beta_weights(theta1, theta2, J):
    """
    Compute normalized Beta polynomial weights for MIDAS.
    
    Parameters
    ----------
    theta1 : float
        First Beta parameter (controls recent lag weight)
    theta2 : float
        Second Beta parameter (controls distant lag weight)
    J : int
        Number of high-frequency lags
    
    Returns
    -------
    weights : ndarray, shape (J,)
        Normalized weights summing to 1
    """
    # Step 1: Create lag indices normalized to [0, 1]
    j = np.arange(J)
    j_normalized = j / J
    
    # Step 2: Compute unnormalized Beta weights
    # Avoid numerical issues with small values
    weights_raw = ((1 - j_normalized)**(theta1 - 1) * 
                   j_normalized**(theta2 - 1))
    
    # Step 3: Normalize to sum to 1
    weights = weights_raw / weights_raw.sum()
    
    return weights


# Visualize different Beta weight specifications
J = 12  # 12 monthly lags for quarterly forecasting
lags = np.arange(J)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Specification 1: Recent lags weighted heavily (typical for nowcasting)
w1 = beta_weights(theta1=5, theta2=1, J=J)
axes[0, 0].bar(lags, w1, alpha=0.7, color='steelblue')
axes[0, 0].set_title(r'Recent Weighted: $\theta_1=5, \theta_2=1$')
axes[0, 0].set_xlabel('Lag (months)')
axes[0, 0].set_ylabel('Weight')
axes[0, 0].grid(True, alpha=0.3)

# Specification 2: Distant lags weighted heavily
w2 = beta_weights(theta1=1, theta2=5, J=J)
axes[0, 1].bar(lags, w2, alpha=0.7, color='coral')
axes[0, 1].set_title(r'Distant Weighted: $\theta_1=1, \theta_2=5$')
axes[0, 1].set_xlabel('Lag (months)')
axes[0, 1].set_ylabel('Weight')
axes[0, 1].grid(True, alpha=0.3)

# Specification 3: Hump-shaped (peak in middle)
w3 = beta_weights(theta1=3, theta2=3, J=J)
axes[1, 0].bar(lags, w3, alpha=0.7, color='darkgreen')
axes[1, 0].set_title(r'Hump-Shaped: $\theta_1=3, \theta_2=3$')
axes[1, 0].set_xlabel('Lag (months)')
axes[1, 0].set_ylabel('Weight')
axes[1, 0].grid(True, alpha=0.3)

# Specification 4: Exponential decay (approximate)
w4 = beta_weights(theta1=2, theta2=1, J=J)
axes[1, 1].bar(lags, w4, alpha=0.7, color='purple')
axes[1, 1].set_title(r'Exponential-like: $\theta_1=2, \theta_2=1$')
axes[1, 1].set_xlabel('Lag (months)')
axes[1, 1].set_ylabel('Weight')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Beta polynomial weights demonstration:")
print(f"All weight schemes sum to 1.0")
print(f"Scheme 1 sum: {w1.sum():.6f}")
print(f"Scheme 3 sum: {w3.sum():.6f}")

### Why Beta Weights?

**Advantages:**
1. **Parsimony:** 2 parameters summarize entire lag distribution (vs. $J$ unrestricted parameters)
2. **Flexibility:** Can approximate many lag patterns
3. **Smoothness:** Enforces gradual weight decay/increase
4. **Economic interpretation:** Reflects information decay over time

**Alternative weighting schemes:**
- Exponential Almon polynomial: $w(j) \propto \exp(\theta_1 j + \theta_2 j^2)$
- Step function: Equal weights within windows
- Unrestricted: Estimate all $J$ weights (not recommended for small samples)

## 1.2 MIDAS Regression Function

The MIDAS forecast combines weighted lags:

$$\hat{y}_t = \beta_0 + \beta_1 \sum_{j=0}^{J-1} w(j; \theta_1, \theta_2) x_{t-j}$$

This can be rewritten as a single weighted regressor:
$$\tilde{x}_t(\theta) = \sum_{j=0}^{J-1} w(j; \theta) x_{t-j}$$

Then: $\hat{y}_t = \beta_0 + \beta_1 \tilde{x}_t(\theta)$

In [ ]:
# Purpose: Create weighted MIDAS regressor
# Key Concept: Compress J lags into single weighted variable

def create_midas_regressor(x_high_freq, theta1, theta2, J):
    """
    Create MIDAS weighted regressor from high-frequency data.
    
    Parameters
    ----------
    x_high_freq : ndarray, shape (T,)
        High-frequency predictor series
    theta1, theta2 : float
        Beta polynomial parameters
    J : int
        Number of lags to include
    
    Returns
    -------
    x_midas : ndarray, shape (T-J+1,)
        Weighted MIDAS regressor (shortened due to lags)
    """
    T = len(x_high_freq)
    
    # Step 1: Compute Beta weights
    weights = beta_weights(theta1, theta2, J)
    
    # Step 2: Create lagged matrix
    # Each row contains [x_t, x_{t-1}, ..., x_{t-J+1}]
    x_lagged = np.zeros((T - J + 1, J))
    for i in range(T - J + 1):
        x_lagged[i, :] = x_high_freq[i:i+J][::-1]  # Reverse to get [x_t, x_{t-1}, ...]
    
    # Step 3: Apply weights (weighted sum across lags)
    x_midas = x_lagged @ weights
    
    return x_midas


# Example: Create MIDAS regressor from monthly data
T_monthly = 60  # 5 years of monthly data
x_monthly = np.sin(np.linspace(0, 4*np.pi, T_monthly)) + np.random.randn(T_monthly) * 0.2

# Create MIDAS regressors with different weights
x_midas_recent = create_midas_regressor(x_monthly, theta1=5, theta2=1, J=12)
x_midas_hump = create_midas_regressor(x_monthly, theta1=3, theta2=3, J=12)

# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 5))
time_midas = np.arange(11, T_monthly)  # Start at lag 11 (J-1)
ax.plot(x_monthly, 'o-', alpha=0.5, label='Original Monthly Data', markersize=4)
ax.plot(time_midas, x_midas_recent, 's-', linewidth=2, label='MIDAS (Recent Weighted)', markersize=6)
ax.plot(time_midas, x_midas_hump, '^-', linewidth=2, label='MIDAS (Hump Weighted)', markersize=6)
ax.set_xlabel('Month')
ax.set_ylabel('Value')
ax.set_title('MIDAS Weighted Regressors vs Original Series')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Original monthly series length: {len(x_monthly)}")
print(f"MIDAS regressor length: {len(x_midas_recent)} (lost {11} observations to lags)")

---

# Part 2: MIDAS Model Estimation

## 2.1 Nonlinear Least Squares

MIDAS is estimated via **nonlinear least squares (NLS)** because the weights $w(j; \theta)$ are nonlinear functions of $\theta$.

**Objective function:**
$$\min_{\beta_0, \beta_1, \theta_1, \theta_2} \sum_{t=1}^T \left(y_t - \beta_0 - \beta_1 \sum_{j=0}^{J-1} w(j; \theta) x_{t-j}\right)^2$$

**Estimation procedure:**
1. Initialize $\theta$ (e.g., $\theta_1 = \theta_2 = 1$)
2. For given $\theta$, compute $\tilde{x}_t(\theta)$
3. Run OLS: $y_t = \beta_0 + \beta_1 \tilde{x}_t(\theta) + \varepsilon_t$ to get $\hat{\beta}(\theta)$
4. Optimize over $\theta$ to minimize SSR
5. Report $\hat{\beta}, \hat{\theta}$

In [ ]:
# Purpose: Implement MIDAS NLS estimation
# Key Concept: Nested optimization - OLS inside nonlinear optimization over theta

class MIDASRegression:
    """
    MIDAS Regression with Beta polynomial weights.
    
    Estimates y_t = beta_0 + beta_1 * sum_j w(j; theta) x_{t-j} + epsilon_t
    """
    
    def __init__(self, J=12):
        """
        Parameters
        ----------
        J : int
            Number of high-frequency lags to include
        """
        self.J = J
        self.beta0_ = None
        self.beta1_ = None
        self.theta1_ = None
        self.theta2_ = None
        self.weights_ = None
        self.ssr_ = None
    
    def _objective(self, theta, y_low, x_high):
        """
        Compute sum of squared residuals for given theta.
        
        Parameters
        ----------
        theta : array-like, shape (2,)
            [theta1, theta2]
        y_low : ndarray
            Low-frequency target variable
        x_high : ndarray
            High-frequency predictor
        
        Returns
        -------
        ssr : float
            Sum of squared residuals
        """
        theta1, theta2 = theta
        
        # Ensure positive parameters
        if theta1 <= 0 or theta2 <= 0:
            return 1e10
        
        try:
            # Step 1: Create MIDAS regressor with current theta
            x_midas = create_midas_regressor(x_high, theta1, theta2, self.J)
            
            # Step 2: Align y and x (both must have same length)
            T_midas = len(x_midas)
            y_aligned = y_low[-T_midas:]
            
            # Step 3: Run OLS regression
            X_reg = np.column_stack([np.ones(T_midas), x_midas])
            beta = np.linalg.lstsq(X_reg, y_aligned, rcond=None)[0]
            
            # Step 4: Compute residuals and SSR
            y_pred = X_reg @ beta
            residuals = y_aligned - y_pred
            ssr = np.sum(residuals**2)
            
            return ssr
        
        except:
            return 1e10
    
    def fit(self, y_low, x_high, theta_init=None):
        """
        Estimate MIDAS model via NLS.
        
        Parameters
        ----------
        y_low : ndarray, shape (T_low,)
            Low-frequency target variable
        x_high : ndarray, shape (T_high,)
            High-frequency predictor (must have T_high >= T_low * m)
        theta_init : array-like, optional
            Initial guess for [theta1, theta2]
        
        Returns
        -------
        self
        """
        if theta_init is None:
            theta_init = [1.5, 1.5]  # Mild hump shape as default
        
        # Step 1: Optimize theta via Nelder-Mead (derivative-free)
        result = minimize(
            self._objective,
            theta_init,
            args=(y_low, x_high),
            method='Nelder-Mead',
            options={'maxiter': 1000}
        )
        
        self.theta1_, self.theta2_ = result.x
        self.ssr_ = result.fun
        
        # Step 2: Compute final beta estimates
        x_midas = create_midas_regressor(x_high, self.theta1_, self.theta2_, self.J)
        T_midas = len(x_midas)
        y_aligned = y_low[-T_midas:]
        
        X_reg = np.column_stack([np.ones(T_midas), x_midas])
        beta = np.linalg.lstsq(X_reg, y_aligned, rcond=None)[0]
        self.beta0_, self.beta1_ = beta
        
        # Step 3: Store weights
        self.weights_ = beta_weights(self.theta1_, self.theta2_, self.J)
        
        return self
    
    def predict(self, x_high):
        """
        Generate predictions using estimated model.
        
        Parameters
        ----------
        x_high : ndarray
            High-frequency predictor data
        
        Returns
        -------
        y_pred : ndarray
            Predicted values
        """
        x_midas = create_midas_regressor(x_high, self.theta1_, self.theta2_, self.J)
        y_pred = self.beta0_ + self.beta1_ * x_midas
        return y_pred
    
    def summary(self):
        """Print estimation results."""
        print("\nMIDAS Regression Results")
        print("=" * 50)
        print(f"Intercept (beta_0):        {self.beta0_:>10.4f}")
        print(f"MIDAS coefficient (beta_1): {self.beta1_:>10.4f}")
        print(f"Theta_1:                    {self.theta1_:>10.4f}")
        print(f"Theta_2:                    {self.theta2_:>10.4f}")
        print(f"Sum of Squared Residuals:   {self.ssr_:>10.4f}")
        print("=" * 50)


print("MIDASRegression class defined successfully!")

### Exercise 2.1: Estimate MIDAS Model on Simulated Data

**Task:** Generate synthetic quarterly GDP and monthly predictor data, then estimate a MIDAS model.

**Expected Output:** Estimated parameters should recover the true data generating process.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Generate data and estimate MIDAS model
#
# Steps:
# 1. Generate monthly predictor (e.g., AR(1) process)
# 2. Create quarterly target using known Beta weights
# 3. Estimate MIDAS model and compare estimated vs true weights

# Step 1: Generate monthly predictor (60 months = 5 years)
np.random.seed(123)
T_m = 60
x_monthly = np.zeros(T_m)
x_monthly[0] = np.random.randn()

# Create AR(1) process: x_t = 0.7 * x_{t-1} + epsilon_t
for t in range(1, T_m):
    x_monthly[t] = None  # Replace with AR(1) formula

# Step 2: Generate quarterly GDP using TRUE Beta weights
true_theta1, true_theta2 = 3.0, 1.5
true_beta0, true_beta1 = 2.0, 0.5
J = 12

# Create true MIDAS regressor
x_midas_true = None  # Replace with create_midas_regressor call

# Generate quarterly GDP (skip first few to align with MIDAS lags)
# Sample quarterly: take every 3rd observation
T_q = len(x_midas_true) // 3
y_quarterly = np.zeros(T_q)
for i in range(T_q):
    # Use MIDAS value from 3rd month of each quarter
    idx = (i + 1) * 3 - 1
    if idx < len(x_midas_true):
        y_quarterly[i] = true_beta0 + true_beta1 * x_midas_true[idx] + np.random.randn() * 0.1

# Step 3: Estimate MIDAS model
midas = MIDASRegression(J=12)
midas.fit(y_quarterly, x_monthly, theta_init=[2.0, 2.0])
midas.summary()

# Step 4: Compare true vs estimated weights
true_weights = beta_weights(true_theta1, true_theta2, J)
estimated_weights = midas.weights_

print(f"\nTrue theta: ({true_theta1}, {true_theta2})")
print(f"Estimated theta: ({midas.theta1_:.2f}, {midas.theta2_:.2f})")

<details>
<summary>Hint 1: AR(1) Process</summary>
The AR(1) formula is: x_monthly[t] = 0.7 * x_monthly[t-1] + np.random.randn()
</details>

<details>
<summary>Hint 2: MIDAS Regressor</summary>
Use: x_midas_true = create_midas_regressor(x_monthly, true_theta1, true_theta2, J)
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    assert x_monthly is not None, "Don't forget to generate monthly data!"
    assert x_midas_true is not None, "Don't forget to create MIDAS regressor!"
    assert len(y_quarterly) > 0, "Quarterly target should have observations!"
    
    assert midas.theta1_ is not None, "MIDAS model should be fitted!"
    assert 0.5 < midas.theta1_ < 10, "Theta1 should be reasonable"
    assert 0.5 < midas.theta2_ < 10, "Theta2 should be reasonable"
    
    # Check that estimated thetas are close to true (within 50% - DGP has noise)
    theta_error = np.abs(midas.theta1_ - true_theta1) + np.abs(midas.theta2_ - true_theta2)
    assert theta_error < 3.0, "Estimated thetas should be reasonably close to true values"
    
    print("✓ Exercise 2.1 passed! MIDAS model estimated successfully.")

test_exercise_2_1()

In [ ]:
# SOLUTION
# --------
np.random.seed(123)
T_m = 60
x_monthly_sol = np.zeros(T_m)
x_monthly_sol[0] = np.random.randn()

for t in range(1, T_m):
    x_monthly_sol[t] = 0.7 * x_monthly_sol[t-1] + np.random.randn()

true_theta1, true_theta2 = 3.0, 1.5
true_beta0, true_beta1 = 2.0, 0.5
J = 12

x_midas_true_sol = create_midas_regressor(x_monthly_sol, true_theta1, true_theta2, J)

T_q = len(x_midas_true_sol) // 3
y_quarterly_sol = np.zeros(T_q)
for i in range(T_q):
    idx = (i + 1) * 3 - 1
    if idx < len(x_midas_true_sol):
        y_quarterly_sol[i] = true_beta0 + true_beta1 * x_midas_true_sol[idx] + np.random.randn() * 0.1

## 2.2 Visualizing Weight Estimates

Compare true vs estimated weight distributions to assess recovery.

In [ ]:
# Purpose: Visualize true vs estimated MIDAS weights
# Key Concept: Good estimation should recover weight pattern

lags = np.arange(J)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(lags - 0.2, true_weights, width=0.4, alpha=0.7, label='True Weights', color='steelblue')
ax.bar(lags + 0.2, midas.weights_, width=0.4, alpha=0.7, label='Estimated Weights', color='coral')
ax.set_xlabel('Lag (months)')
ax.set_ylabel('Weight')
ax.set_title('MIDAS Weight Recovery: True vs Estimated')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Compute weight correlation
weight_corr = np.corrcoef(true_weights, midas.weights_)[0, 1]
print(f"Correlation between true and estimated weights: {weight_corr:.3f}")

---

# Part 3: GDP Forecasting with MIDAS

## 3.1 Realistic Macroeconomic Data

We'll generate realistic GDP and Industrial Production data to demonstrate MIDAS forecasting.

In [ ]:
# Purpose: Generate realistic GDP and IP data with business cycle properties
# Key Concept: Monthly IP helps forecast quarterly GDP

np.random.seed(2024)

# Generate 10 years of monthly data (120 months)
T_months = 120
time_monthly = np.arange(T_months)

# Create monthly Industrial Production Index
# Trend + cycle + noise
trend = 100 + 0.15 * time_monthly
cycle = 5 * np.sin(2 * np.pi * time_monthly / 48)  # 4-year cycle
noise = np.random.randn(T_months) * 1.5
ip_monthly = trend + cycle + noise

# Create quarterly GDP (more aggregated, less volatile)
# GDP depends on IP with lag structure
T_quarters = 40  # 10 years quarterly
gdp_quarterly = np.zeros(T_quarters)

for q in range(T_quarters):
    # Quarter q corresponds to months [3q, 3q+1, 3q+2]
    month_indices = [3*q, 3*q+1, 3*q+2]
    
    # GDP depends on average IP in quarter plus some idiosyncratic component
    if max(month_indices) < T_months:
        avg_ip = ip_monthly[month_indices].mean()
        gdp_quarterly[q] = 0.95 * avg_ip + np.random.randn() * 0.8

# Compute GDP growth rates (typical target for forecasting)
gdp_growth = np.diff(gdp_quarterly) / gdp_quarterly[:-1] * 100  # Percent growth
ip_growth = np.diff(ip_monthly) / ip_monthly[:-1] * 100

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Levels
axes[0].plot(time_monthly, ip_monthly, label='IP (Monthly)', alpha=0.7, linewidth=1.5)
axes[0].plot(np.arange(T_quarters) * 3 + 1, gdp_quarterly, 'o-', 
             label='GDP (Quarterly)', linewidth=2, markersize=6)
axes[0].set_ylabel('Index Level')
axes[0].set_title('Industrial Production (Monthly) and GDP (Quarterly) - Levels')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Growth rates
axes[1].plot(time_monthly[1:], ip_growth, label='IP Growth (Monthly)', alpha=0.6, linewidth=1)
axes[1].plot(np.arange(len(gdp_growth)) * 3 + 1, gdp_growth, 'o-', 
             label='GDP Growth (Quarterly)', linewidth=2, markersize=6, color='darkred')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Growth Rate (%)')
axes[1].set_title('Growth Rates')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Generated {T_months} months of IP data")
print(f"Generated {T_quarters} quarters of GDP data")
print(f"Correlation (levels): {np.corrcoef(ip_monthly[::3][:len(gdp_quarterly)], gdp_quarterly)[0,1]:.3f}")

## 3.2 Out-of-Sample Forecast Evaluation

We'll use **expanding window** cross-validation:
1. Train on data up to quarter $t$
2. Forecast quarter $t+1$ GDP growth
3. Compare to actual value
4. Expand window and repeat

This mimics real-time forecasting conditions.

In [ ]:
# Purpose: Implement expanding window forecast evaluation
# Key Concept: Proper out-of-sample evaluation avoids look-ahead bias

def expanding_window_forecast(y_quarterly, x_monthly, min_train=20, J=12):
    """
    Generate out-of-sample forecasts using expanding window.
    
    Parameters
    ----------
    y_quarterly : ndarray
        Quarterly target variable
    x_monthly : ndarray
        Monthly predictor
    min_train : int
        Minimum training sample size (quarters)
    J : int
        MIDAS lag length
    
    Returns
    -------
    forecasts : list
        Out-of-sample forecasts
    actuals : list
        Actual values
    """
    T_q = len(y_quarterly)
    forecasts = []
    actuals = []
    
    for t in range(min_train, T_q - 1):
        # Training data: quarters 0 to t
        y_train = y_quarterly[:t+1]
        
        # Corresponding monthly data (need 3*t + J months minimum)
        x_train = x_monthly[:3*t + J + 3]
        
        try:
            # Fit MIDAS model
            midas = MIDASRegression(J=J)
            midas.fit(y_train, x_train, theta_init=[2.0, 1.0])
            
            # Forecast next quarter using data up to end of quarter t+1
            x_forecast = x_monthly[:3*(t+2) + J]
            y_pred = midas.predict(x_forecast)
            
            forecasts.append(y_pred[-1])  # Last prediction is for quarter t+1
            actuals.append(y_quarterly[t+1])
        
        except:
            continue
    
    return np.array(forecasts), np.array(actuals)


# Run expanding window forecasts for GDP growth
print("Running expanding window forecast evaluation...")
print("This may take 30-60 seconds...\n")

forecasts_midas, actuals = expanding_window_forecast(
    gdp_growth, ip_growth, min_train=15, J=12
)

# Compute forecast errors
rmse = np.sqrt(mean_squared_error(actuals, forecasts_midas))
mae = mean_absolute_error(actuals, forecasts_midas)

print(f"Out-of-Sample Forecast Performance:")
print(f"  RMSE: {rmse:.3f}%")
print(f"  MAE:  {mae:.3f}%")
print(f"  Number of forecasts: {len(forecasts_midas)}")

### Visualize Forecast Performance

In [ ]:
# Purpose: Visualize actual vs forecast GDP growth
# Key Concept: Good forecasts should track turning points

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Plot 1: Actual vs Forecast
forecast_quarters = np.arange(len(forecasts_midas))
axes[0].plot(forecast_quarters, actuals, 'o-', label='Actual GDP Growth', 
             linewidth=2, markersize=7, color='steelblue')
axes[0].plot(forecast_quarters, forecasts_midas, 's--', label='MIDAS Forecast', 
             linewidth=2, markersize=6, color='coral', alpha=0.8)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('GDP Growth (%)')
axes[0].set_title('Out-of-Sample GDP Growth Forecasts: MIDAS vs Actual')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Forecast Errors
errors = actuals - forecasts_midas
axes[1].bar(forecast_quarters, errors, alpha=0.7, color='darkred')
axes[1].axhline(0, color='black', linewidth=1.5)
axes[1].axhline(rmse, color='orange', linestyle='--', linewidth=2, label=f'RMSE = {rmse:.2f}%')
axes[1].axhline(-rmse, color='orange', linestyle='--', linewidth=2)
axes[1].set_xlabel('Forecast Period (Quarter)')
axes[1].set_ylabel('Forecast Error (%)')
axes[1].set_title('Forecast Errors')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Error distribution
print(f"\nError Distribution:")
print(f"  Mean Error: {errors.mean():.3f}% (should be near 0)")
print(f"  Std Error:  {errors.std():.3f}%")
print(f"  Min Error:  {errors.min():.3f}%")
print(f"  Max Error:  {errors.max():.3f}%")

### Exercise 3.1: Compare MIDAS to Naive Benchmark

**Task:** Implement a naive benchmark (e.g., random walk or AR(1) on GDP growth only) and compare to MIDAS.

**Expected Output:** MIDAS should outperform the naive benchmark by utilizing monthly IP information.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Implement AR(1) benchmark and compare to MIDAS
#
# Steps:
# 1. For each forecast period, estimate AR(1): y_t = alpha + beta * y_{t-1} + epsilon
# 2. Forecast next quarter using AR(1)
# 3. Compute RMSE and compare to MIDAS

def ar1_expanding_window_forecast(y_quarterly, min_train=20):
    """
    Generate AR(1) forecasts using expanding window.
    
    Parameters
    ----------
    y_quarterly : ndarray
        Quarterly target variable
    min_train : int
        Minimum training sample size
    
    Returns
    -------
    forecasts : ndarray
        Out-of-sample forecasts
    actuals : ndarray
        Actual values
    """
    T_q = len(y_quarterly)
    forecasts = []
    actuals = []
    
    for t in range(min_train, T_q - 1):
        # Training data
        y_train = y_quarterly[:t+1]
        
        # Create lagged variable
        y_lag = y_train[:-1]
        y_current = y_train[1:]
        
        # Estimate AR(1): y_t = alpha + beta * y_{t-1}
        X_ar = np.column_stack([np.ones(len(y_lag)), y_lag])
        beta_ar = None  # Replace with np.linalg.lstsq(X_ar, y_current, rcond=None)[0]
        
        # Forecast: y_{t+1} = alpha + beta * y_t
        y_pred = None  # Replace with beta_ar[0] + beta_ar[1] * y_quarterly[t]
        
        forecasts.append(y_pred)
        actuals.append(y_quarterly[t+1])
    
    return np.array(forecasts), np.array(actuals)


# Generate AR(1) forecasts
forecasts_ar1, actuals_ar1 = ar1_expanding_window_forecast(gdp_growth, min_train=15)

# Compute metrics
rmse_ar1 = np.sqrt(mean_squared_error(actuals_ar1, forecasts_ar1))
mae_ar1 = mean_absolute_error(actuals_ar1, forecasts_ar1)

print("\nForecast Comparison:")
print("=" * 50)
print(f"AR(1) Benchmark:")
print(f"  RMSE: {rmse_ar1:.3f}%")
print(f"  MAE:  {mae_ar1:.3f}%")
print(f"\nMIDAS:")
print(f"  RMSE: {rmse:.3f}%")
print(f"  MAE:  {mae:.3f}%")
print(f"\nImprovement:")
print(f"  RMSE reduction: {(rmse_ar1 - rmse) / rmse_ar1 * 100:.1f}%")
print(f"  MAE reduction:  {(mae_ar1 - mae) / mae_ar1 * 100:.1f}%")
print("=" * 50)

<details>
<summary>Hint 1: OLS Estimation</summary>
Use: beta_ar = np.linalg.lstsq(X_ar, y_current, rcond=None)[0]
</details>

<details>
<summary>Hint 2: AR(1) Forecast</summary>
The forecast is: y_pred = beta_ar[0] + beta_ar[1] * y_quarterly[t]
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1():
    assert forecasts_ar1 is not None, "Don't forget to generate AR(1) forecasts!"
    assert len(forecasts_ar1) > 0, "AR(1) forecasts should have observations!"
    
    assert rmse_ar1 > 0, "RMSE should be computed!"
    assert len(forecasts_ar1) == len(actuals_ar1), "Forecasts and actuals should align!"
    
    # MIDAS should typically outperform AR(1) (not guaranteed but expected)
    print(f"\nAR(1) RMSE: {rmse_ar1:.3f}")
    print(f"MIDAS RMSE: {rmse:.3f}")
    
    if rmse < rmse_ar1:
        print("✓ MIDAS outperforms AR(1) benchmark!")
    else:
        print("Note: AR(1) performed better in this sample (can happen with random data)")
    
    print("✓ Exercise 3.1 passed! Benchmark comparison completed.")

test_exercise_3_1()

In [ ]:
# SOLUTION
# --------
def ar1_expanding_window_forecast_solution(y_quarterly, min_train=20):
    T_q = len(y_quarterly)
    forecasts = []
    actuals = []
    
    for t in range(min_train, T_q - 1):
        y_train = y_quarterly[:t+1]
        y_lag = y_train[:-1]
        y_current = y_train[1:]
        
        X_ar = np.column_stack([np.ones(len(y_lag)), y_lag])
        beta_ar = np.linalg.lstsq(X_ar, y_current, rcond=None)[0]
        
        y_pred = beta_ar[0] + beta_ar[1] * y_quarterly[t]
        
        forecasts.append(y_pred)
        actuals.append(y_quarterly[t+1])
    
    return np.array(forecasts), np.array(actuals)

---

# Summary and Key Takeaways

## What You've Learned

1. **MIDAS Framework**
   - Forecasts low-frequency variables using high-frequency predictors
   - Avoids information loss from temporal aggregation
   - Parsimoniously weights lags via Beta polynomial

2. **Beta Polynomial Weights**
   - Two parameters control entire lag distribution
   - Flexible enough to capture various decay patterns
   - Economically interpretable (information decay over time)

3. **Nonlinear Least Squares Estimation**
   - MIDAS requires optimization over theta parameters
   - Nested OLS within nonlinear optimization
   - Convergence depends on good starting values

4. **Forecast Evaluation**
   - Expanding window mimics real-time forecasting
   - MIDAS typically outperforms frequency-matched benchmarks
   - Monthly indicators contain valuable information for quarterly forecasts

## Connection to Mixed-Frequency DFMs

MIDAS is a **reduced-form** approach to mixed-frequency forecasting. In the next notebook, we'll see the **structural** approach: mixed-frequency Dynamic Factor Models that:
- Explicitly model latent factors at high frequency
- Handle multiple predictors simultaneously via factor structure
- Use state-space methods (Kalman filter) for estimation
- Naturally handle ragged-edge data

MIDAS and MF-DFM are complementary:
- MIDAS: Simple, fast, good for single-predictor forecasts
- MF-DFM: More complex, handles many predictors, unified framework

## Next Steps

You're now ready to:
1. Proceed to **Notebook 5.2: GDP Nowcasting** - Apply MIDAS to real-time GDP nowcasting
2. Learn about mixed-frequency state-space models (Module 5 Guides 3-4)
3. Implement complete nowcasting system with ragged-edge data

---

## Self-Assessment

Before moving on, ensure you can:
- [ ] Compute Beta polynomial weights for given theta parameters
- [ ] Create MIDAS weighted regressors from high-frequency data
- [ ] Estimate MIDAS models via NLS
- [ ] Implement expanding window forecast evaluation
- [ ] Compare MIDAS to benchmark models
- [ ] Interpret estimated lag weights economically

## Additional Resources

- **Ghysels et al. (2004):** "The MIDAS Touch" - Original MIDAS paper
- **Foroni et al. (2015):** "Unrestricted MIDAS" - Extensions and variations
- **Clements & Galvao (2008):** "Macroeconomic forecasting with mixed-frequency data"
- **Python package:** midaspy - MIDAS estimation tools

---

*You've successfully implemented MIDAS regression and applied it to GDP forecasting. The Beta polynomial weighting scheme provides a parsimonious yet flexible way to exploit high-frequency information for low-frequency forecasts.*